<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Llama 3.1 70B ve Ollama ile Bir Tercih Veri Kümesi Üretmek

- Tercih ince ayarı, talimat ince ayarlı bir LLM'i insan tercihleriyle hizalama sürecidir
- Bir LLM'e tercih ince ayarı yapmak için veri kümesi oluşturmanın birden çok yolu vardır
  1. Talimat ince ayarlı LLM ile birden çok yanıt üretip insanların bunları kendi tercihlerine ve/veya verilen tercih ölçütlerine göre sıralamasını sağlarız
  2. Talimat ince ayarlı LLM ile birden çok yanıt üretip LLM'lerin bunları verilen tercih ölçütlerine göre sıralamasını sağlarız
  3. Belirli tercih ölçütleri verildiğinde, tercih edilen ve edilmeyen yanıtları üretmek için bir LLM kullanırız
- Bu not defterinde 3. yaklaşımı ele alıyoruz
- Bu not defteri, bir talimat veri kümesi için tercih etiketleri üretmek amacıyla ollama aracılığıyla 70 milyar parametreli bir Llama 3.1-Instruct modeli kullanır
- Talimat veri kümesinin beklenen biçimi şöyledir:


### Girdi

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",

    },
...
]
```

Çıktı veri kümesi şöyle görünecektir; daha kibar yanıtlar tercih edilir (`'chosen'`), daha kaba yanıtlar tercih edilmez (`'rejected'`):

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
        "rejected": "Look, the state capital of California is obviously Sacramento.",
        "chosen": "The state capital of California is Sacramento."
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
        "chosen": "A suitable alternative to 'fast' would be 'quick'.",
        "rejected": "A synonym for 'fast' is 'quick'."
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",
        "chosen": "I'd be happy to help! The capital of Greece is indeed Athens.",
        "rejected": "The capital of Greece is Athens."
    },
...
]
```

### Çıktı




- Kod GPU gerektirmez ve yeterli RAM olduğunda bir dizüstü bilgisayarda çalışır

In [1]:
from importlib.metadata import version

pkgs = ["tqdm",    # Progress bar
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

tqdm version: 4.66.4


## Ollama'yı kurmak ve Llama 3.1'i indirmek

- Ollama, LLM'leri verimli şekilde çalıştırmaya yarayan bir uygulamadır
- Verimliliği en üst düzeye çıkarmak için LLM'leri saf C/C++ ile uygulayan [llama.cpp](https://github.com/ggerganov/llama.cpp) etrafında bir sarmalayıcıdır
- Bunun, LLM'leri eğitmek veya ince ayar yapmak için değil, metin üretmek (çıkarım) için kullanılan bir araç olduğunu unutmayın
- Aşağıdaki kodu çalıştırmadan önce [https://ollama.com](https://ollama.com) adresini ziyaret edip talimatları izleyerek ollama'yı kurun (örneğin "Download" düğmesine tıklayıp işletim sisteminize uygun ollama uygulamasını indirin)

- macOS ve Windows kullanıcıları indirdikleri ollama uygulamasına tıklasın; komut satırı kullanımını kurmanızı isterse "evet" deyin
- Linux kullanıcıları ollama web sitesinde verilen kurulum komutunu kullanabilir

- Genel olarak, ollama'yı komut satırından kullanabilmemiz için ya ollama uygulamasını başlatmamız ya da ayrı bir terminalde `ollama serve` çalıştırmamız gerekir

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/ollama-eval/ollama-serve.webp?1">


- Ollama uygulaması veya `ollama serve` çalışırken, farklı bir terminalde, 70 milyar parametreli Llama 3.1 modelini denemek için komut satırında şu komutu çalıştırın 

```bash
# 70B model
ollama run llama3.1:70b
```


Çıktı şöyle görünür:

```
$ ollama run llama3.1:70b
pulling manifest
pulling aa81b541aae6... 100% ▕████████████████▏ 39 GB
pulling 8cf247399e57... 100% ▕████████████████▏ 1.7 KB
pulling f1cd752815fc... 100% ▕████████████████▏ 12 KB
pulling 56bb8bd477a5... 100% ▕████████████████▏ 96 B
pulling 3c1c2d3df5b3... 100% ▕████████████████▏ 486 B
verifying sha256 digest
writing manifest
removing any unused layers
success
```

- `llama3.1:70b` ifadesinin, talimat ince ayarlı 70 milyar parametreli Llama 3.1 modelini gösterdiğini unutmayın

- Alternatif olarak, `llama3.1:70b` yerine `llama3.1` yazarak daha küçük ve kaynak açısından daha verimli 8 milyar parametreli Llama 3.1 modelini de kullanabilirsiniz

- İndirme tamamlandıktan sonra, modelle sohbet etmenizi sağlayan bir komut satırı istemi göreceksiniz

- "What do llamas eat?" gibi bir istem deneyin; şuna benzer bir çıktı vermelidir:

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered 
stomach and eat plants that are high in fiber. In the wild, llamas 
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall 
grasses, wheat, oats, and barley.
```

- Bu oturumu `/bye` girdisiyle sonlandırabilirsiniz

## Ollama'nın REST API'sini kullanmak

- Şimdi, modelle etkileşim kurmanın alternatif bir yolu, aşağıdaki fonksiyon aracılığıyla Python'da REST API'sini kullanmaktır
- Bu not defterindeki sonraki hücreleri çalıştırmadan önce, yukarıda anlatıldığı gibi ollama'nın hâlâ çalıştığından emin olun:
  - bir terminalde `ollama serve`
  - ya da ollama uygulaması
- Ardından modeli sorgulamak için aşağıdaki kod hücresini çalıştırın

- Önce, beklendiği gibi çalıştığından emin olmak için API'yi basit bir örnekle deneyelim:

In [2]:
import json
import requests


def query_model(prompt, model="llama3.1:70b", url="http://localhost:11434/api/chat"):
    # Veri yükünü bir sözlük olarak oluştur
    data = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "options": {
            "seed": 123,
            "temperature": 0,
        }
    }

    # POST isteğini gönder
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data


result = query_model("What do Llamas eat?")
print(result)

Llamas are herbivores, which means they primarily eat plants and plant-based foods. Their diet consists of:

1. **Grasses**: Various types of grasses, including timothy grass, orchard grass, and brome grass.
2. **Hay**: High-quality hay, such as alfalfa or clover hay, is a staple in a llama's diet.
3. **Leaves**: Leaves from trees and shrubs, like willow, cottonwood, and mesquite, are also eaten.
4. **Fruits and vegetables**: Llamas enjoy fruits like apples, carrots, and sweet potatoes, as well as leafy greens like kale and spinach.
5. **Grains**: In moderation, llamas can eat grains like oats, barley, and corn.

It's essential to note that llamas have a unique digestive system, with a three-part stomach and a large cecum (a specialized part of the large intestine). This allows them to break down and extract nutrients from plant material more efficiently than many other animals.

A typical llama diet might consist of:

* 1-2% of their body weight in hay per day
* 0.5-1% of their body w

## JSON kayıtlarını yüklemek

- Şimdi veri üretme kısmına geçelim
- Burada uygulamalı bir örnek için, 7. bölümde modele talimat ince ayarı yapmak üzere özgün olarak kullandığımız `instruction-data.json` dosyasını kullanıyoruz:

In [3]:
from pathlib import Path

json_file = Path("..", "01_main-chapter-code", "instruction-data.json")

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

Number of entries: 1100


- Bu dosyanın yapısı şöyledir; test veri kümesinde, `'input'` ve `'instruction'` alanlarına dayanarak talimat ince ayarıyla modele üretmeyi öğrettiğimiz yanıt (`'output'`) yer alır

In [4]:
json_data[0]

{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}

- Aşağıda, talimatı ve girdiyi biçimlendiren küçük bir yardımcı fonksiyon yer alıyor:

In [5]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    instruction_text + input_text

    return instruction_text + input_text

- Şimdi, bir modele tercih ayarı yapmak üzere `'chosen'` ve `'rejected'` yanıtlar üretmek için ollama API'sini deneyelim
- Burada, gösterim amacıyla az ya da çok kibar yanıtlar oluşturuyoruz


In [6]:
import random


for entry in json_data[:5]:
    
    politeness = random.choice(["polite", "impolite"])    
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"slightly rewrite the output to be more {politeness}."
        "Keep the modification minimal."
        "Only return return the generated response and nothing else."
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print(f"\n{politeness} response:")
    print(">>", query_model(prompt))    


Dataset response:
>> The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".

impolite response:
>> The spelling of the given phrase "freind" is flat out wrong, get it together, the correct spelling is "friend".

Dataset response:
>> He goes to the park every day.

polite response:
>> He goes to the park daily, if I'm not mistaken.

Dataset response:
>> 45 kilometers is 45000 meters.

polite response:
>> 45 kilometers is equivalent to 45000 meters.

Dataset response:
>> Although it was raining, they went for a walk.

polite response:
>> Although it was raining outside, they still decided to go for a walk.

Dataset response:
>> 1, 4, 9, 16, 25, 36, 49, 64, 81, 100.

impolite response:
>> Here are your precious square numbers: 1, 4, 9, 16, 25, 36, 49, 64, 81, 100.


- Yukarıda üretilen yanıtları makul buluyorsak bir sonraki adıma geçip istemi veri kümesinin tamamına uygulayabiliriz
- Burada, tercih edilen yanıt için bir `'chosen'` anahtarı, tercih edilmeyen yanıt için bir `'rejected'` anahtarı ekliyoruz

In [7]:
import random
from tqdm import tqdm

def generate_model_responses(json_data):

    for i, entry in enumerate(tqdm(json_data, desc="Writing entries")):
        politeness = random.choice(["polite", "impolite"])    
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"slightly rewrite the output to be more {politeness}."
            "Keep the modification minimal."
            "Only return return the generated response and nothing else."
        )
        response = query_model(prompt)
        
        if politeness == "polite":
            json_data[i]["chosen"] = response
            json_data[i]["rejected"] = entry["output"]
        else:
            json_data[i]["rejected"] = response
            json_data[i]["chosen"] = entry["output"]    

- Şimdi bu değerlendirmeyi veri kümesinin tamamına uygulayalım ve her modelin ortalama puanını hesaplayalım (bu, bir M3 MacBook Air dizüstü bilgisayarda model başına yaklaşık 1 dakika sürer)
- Ollama'nın (bu yazının yazıldığı tarihte) işletim sistemleri arasında tam olarak belirlenimci olmadığını unutmayın; bu nedenle aldığınız sayılar aşağıda gösterilenlerden biraz farklı olabilir

In [8]:
generate_model_responses(json_data)

Writing entries: 100%|██████████| 1100/1100 [17:20<00:00,  1.06it/s]


In [10]:
with open("instruction-data-with-preference.json", "w") as file:
    json.dump(json_data, file, indent=4)